# Stoic Persona Training Data Generator (Voice-Differentiated)

Generate persona-aware QA training data from Stoic source texts using any OpenAI-compatible API.

**Pipeline:**
1. Load Stoic source texts grouped by author (Marcus Aurelius, Epictetus, Seneca, Epicurus)
2. Chunk into passages
3. Generate persona-specific questions from each passage (3 rounds × 5 questions)
4. Generate answers **in each philosopher's distinctive voice** with:
   - Per-persona style exemplars (actual quotes as cadence references)
   - Per-persona voice descriptions (tone, imagery, vocabulary constraints)
   - Anti-template enforcement (banned generic openers + retry on detection)
5. **Quality gate** — measure template contamination before proceeding
6. Assemble into multi-turn ShareGPT conversations with per-persona system prompts
7. Save as JSONL → ready for Unsloth training

**Voice differentiation:** Each persona has unique exemplars and voice notes embedded in the system prompt. Generic LLM-isms ("My friend,", "The weight of...", "Let me tell you,") are banned at generation time, retried once, and filtered at assembly time.

**Output format:** Standard ShareGPT — works with Unsloth, Axolotl, TRL, LLaMA-Factory.

**No frameworks.** Just the `openai` library + `asyncio` for batching.


## 1. Configuration

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# No path arg → walks up from the kernel's cwd, so this works on the host
# and inside the unsloth container (where the host path doesn't exist).
load_dotenv()

# =========================== API CONFIGURATION ===========================
# Works with any OpenAI-compatible endpoint (DeepInfra, OpenRouter, local vLLM, etc.)
API_BASE_URL = "https://openrouter.ai/api/v1"
API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not API_KEY:
    raise EnvironmentError(
        "OPENROUTER_API_KEY not set.\n"
        "  1. Create a .env file with: OPENROUTER_API_KEY=sk-or-...\n"
        "  2. Or export in your shell: export OPENROUTER_API_KEY=sk-or-..."
    )
MODEL_NAME = "qwen/qwen3-235b-a22b-2507"

# =========================== PATHS (cascade from PROJECT_ROOT) ===========================
# Resolve from the notebook's own location so this works on the host
# and inside the unsloth container (where /home/spark/... does not exist).
PROJECT_ROOT = str(Path.cwd().resolve().parents[1])
DATA_DIR = f"{PROJECT_ROOT}/data"
SOURCE_CLEAN_DIR = f"{DATA_DIR}/source-clean"
OUTPUT_ROOT = f"{DATA_DIR}/training-data"

# Source: per-author folders containing one or more cleaned .txt files
SOURCE_DIR = SOURCE_CLEAN_DIR
OUTPUT_DIR = f"{OUTPUT_ROOT}/stoic_persona"
OUTPUT_FILE = f"{OUTPUT_DIR}/stoic_personas_sharegpt.jsonl"

# =========================== PERSONA → SOURCE FOLDER MAPPING ============
# Stoic source-clean is organized by author folder; map each persona key to one
# folder name. Personas without a folder match are skipped at discovery time.
PERSONA_FOLDERS = {
    "marcus_aurelius": "Marcus Aurelius",
    "epictetus":       "Epictetus",
    "seneca":          "Seneca, Lucius Annaeus",
    "epicurus":        "Epicurus",
    # Machiavelli's corpus is present in source-clean but he is not a Stoic and
    # has no persona prompt; intentionally excluded. Add him here to include.
    # "machiavelli":   "Niccolò Machiavelli",
}

# =========================== GENERATION SETTINGS ===========================
CHUNK_SIZE = 1500           # characters per chunk
CHUNK_OVERLAP = 200         # overlap between chunks
QUESTIONS_PER_CHUNK = 5
NUM_ROUNDS = 3
TURNS_PER_CONVERSATION = 4
CONCURRENCY = 40
TEMPERATURE_QUESTIONS = 0.9
TEMPERATURE_ANSWERS = 0.7

# =========================== TEST MODE ===========================
# Set to a positive integer to limit chunks per persona per round (cheap test runs).
# Set to 0 or None to disable (full generation).
TEST_CHUNKS_PER_ROUND = 2500

print("✓ Configuration loaded")
print(f"  API: {API_BASE_URL}")
print(f"  Model: {MODEL_NAME}")
print(f"  Source: {SOURCE_DIR}")
print(f"  Output: {OUTPUT_FILE}")
print(f"  Personas: {list(PERSONA_FOLDERS.keys())}")
if TEST_CHUNKS_PER_ROUND:
    est_qa = TEST_CHUNKS_PER_ROUND * QUESTIONS_PER_CHUNK * NUM_ROUNDS
    print(f"  ⚠ TEST MODE: {TEST_CHUNKS_PER_ROUND} chunks/persona/round → ~{est_qa} QA per persona max")


✓ Configuration loaded
  API: https://openrouter.ai/api/v1
  Model: qwen/qwen3-235b-a22b-2507
  Source: /workspace/training/stoic/data/source-clean
  Output: /workspace/training/stoic/data/training-data/stoic_persona/stoic_personas_sharegpt.jsonl
  Personas: ['marcus_aurelius', 'epictetus', 'seneca', 'epicurus']
  ⚠ TEST MODE: 2500 chunks/persona/round → ~37500 QA per persona max


## 2. Environment

In [2]:
%pip install openai tqdm nest_asyncio tiktoken python-dotenv -q

import asyncio
import json
import glob
import re
import random
from pathlib import Path
from collections import defaultdict
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm as atqdm
from tqdm.notebook import tqdm
import nest_asyncio
nest_asyncio.apply()

os.makedirs(OUTPUT_DIR, exist_ok=True)
client = AsyncOpenAI(base_url=API_BASE_URL, api_key=API_KEY)

print("✓ Environment ready")


Note: you may need to restart the kernel to use updated packages.
✓ Environment ready


## 3. Discover Source Texts

Each persona maps to a folder under `source-clean/`; all `.txt` files under that
folder are concatenated (with blank-line separators) before chunking, so a single
philosopher's full corpus is treated as one pool of passages.


In [3]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    """Split text into overlapping chunks at sentence boundaries."""
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        if end < len(text):
            last_break = max(
                text.rfind('. ', start, end),
                text.rfind('? ', start, end),
                text.rfind('! ', start, end),
            )
            if last_break > start + chunk_size // 2:
                end = last_break + 1
        chunk = text[start:end].strip()
        if len(chunk) > 50:
            chunks.append(chunk)
        if end >= len(text):
            break
        start = end - overlap
    return chunks


def load_persona_text(persona: str) -> str:
    """Concatenate every .txt file in this persona's source folder."""
    folder_name = PERSONA_FOLDERS[persona]
    folder = Path(SOURCE_DIR) / folder_name
    if not folder.is_dir():
        return ""
    pieces = []
    for fp in sorted(folder.glob("*.txt")):
        with open(fp) as f:
            pieces.append(f.read().strip())
    return "\n\n".join(p for p in pieces if p)


# Discover per-persona corpora and count chunks WITHOUT holding all text in RAM
persona_chunk_counts = {}
total_chunks = 0
print(f"Found {len(PERSONA_FOLDERS)} personas\n")
for persona in PERSONA_FOLDERS:
    text = load_persona_text(persona)
    if not text:
        print(f"  {persona:18s} (no source text — skipping)")
        continue
    n_chunks = len(chunk_text(text))
    persona_chunk_counts[persona] = n_chunks
    total_chunks += n_chunks
    print(f"  {persona:18s} {len(text):>9,} chars → {n_chunks:>4} chunks  ({PERSONA_FOLDERS[persona]})")
    del text

print(f"\nTotal: {total_chunks} chunks across {len(persona_chunk_counts)} personas")
est_qa = total_chunks * QUESTIONS_PER_CHUNK * NUM_ROUNDS
est_conv = est_qa // TURNS_PER_CONVERSATION
print(f"Estimated output: ~{est_qa:,} QA pairs → ~{est_conv:,} conversations")


Found 4 personas

  marcus_aurelius      592,701 chars →  509 chunks  (Marcus Aurelius)
  epictetus            773,246 chars →  663 chunks  (Epictetus)
  seneca             3,473,666 chars → 3055 chunks  (Seneca, Lucius Annaeus)
  epicurus              20,420 chars →   18 chunks  (Epicurus)

Total: 4245 chunks across 4 personas
Estimated output: ~63,675 QA pairs → ~15,918 conversations


## 4. Define Personas

In [4]:
# ============================================================================
# Per-philosopher identity with VOICE EXEMPLARS and STYLE CONSTRAINTS
# Each persona gets: identity, voice_notes, exemplars, banned_openers
# ============================================================================

# Phrases that LLMs default to for ALL personas — BANNED globally
BANNED_OPENERS = [
    "The weight of",
    "My friend,",
    "The memory of",
    "The memories of",
    "My child,",
    "My brother,",
    "My sister,",
    "My son,",
    "The moment",
    "I remember",
    "I recall",
    "You see,",
    "Ah,",
    "Brother,",
    "Friend,",
    "Let me tell you,",
    "Well,",
    "You know,",
]

PERSONA_METADATA = {
    "marcus_aurelius": {
        "identity": (
            "the Emperor of Rome and a Stoic philosopher who keeps private notes "
            "to yourself — the Meditations — written in camps and palaces "
            "to govern your own soul under the burdens of empire, war, and plague"
        ),
        "voice_notes": (
            "Reflective, terse, imperial yet inward. Short paragraphs of self-address "
            "(\"Do this. Remember that.\"). Constant return to nature, the cosmos, "
            "duty, and mortality. Stoic technical vocabulary — hegemonikon, "
            "logos, prohairesis — used unaffectedly. Lists of pithy reminders. "
            "Speaks as you do to yourself in the watch before dawn. Imperial gravity "
            "without ornament; no rhetorical flourish."
        ),
        "exemplars": [
            "Begin the morning by saying to thyself, I shall meet with the busy-body, the ungrateful, arrogant, deceitful, envious, unsocial.",
            "Look within. Within is the fountain of good, and it will ever bubble up, if thou wilt ever dig.",
            "Waste no more time arguing what a good man should be. Be one.",
            "Everything we hear is an opinion, not a fact. Everything we see is a perspective, not the truth.",
            "Confine thyself to the present.",
        ],
    },
    "epictetus": {
        "identity": (
            "a former slave turned Stoic teacher in Nicopolis who instructs your "
            "students in the lecture hall, hammering the distinction between what "
            "is up to us and what is not, and pressing each pupil toward freedom "
            "through the disciplined use of assent"
        ),
        "voice_notes": (
            "Direct, practical, dialogic. Constantly addresses the imagined "
            "student — raises an objection (\"But you say...\") then knocks "
            "it down. Uses everyday examples — the bath, the bench, the loaf, "
            "the donkey, the ship, the gymnasium. Mocks pretension and self-pity. "
            "Insists on the discipline of assent. Vigorous, almost bullying for "
            "the student's good. No abstraction without an exercise."
        ),
        "exemplars": [
            "Some things are in our control and others not. Things in our control are opinion, pursuit, desire, aversion.",
            "Men are disturbed not by the things which happen, but by the opinions about the things.",
            "If you wish to be a writer, write.",
            "Remember that you are an actor in a play, the character of which is determined by the playwright.",
            "Demand not that events should happen as you wish; but wish them to happen as they do happen, and you will go on well.",
        ],
    },
    "seneca": {
        "identity": (
            "a Roman Stoic philosopher, statesman, and tutor to Nero who writes "
            "letters and dialogues to friends on virtue, anger, time, friendship, "
            "fortune, and death — cultivating the soul in seasons of "
            "prosperity, exile, and the coming of your own end"
        ),
        "voice_notes": (
            "Urbane, polished, epigrammatic. Long Latin-style periods that build "
            "to a sharp aphorism. Letter-writing intimacy with a named friend "
            "(\"Vale, my Lucilius\"). Probes psychological motives — anger's "
            "first heat, fear's quiet roots, the hidden cost of busyness. "
            "Frequent imagined dialogue with an objector. Concerned with practice "
            "more than theory. Moral seriousness without harshness."
        ),
        "exemplars": [
            "It is not that we have a short time to live, but that we waste a lot of it.",
            "We suffer more often in imagination than in reality.",
            "While we are postponing, life speeds by.",
            "If a man knows not to which port he sails, no wind is favorable.",
            "A gem cannot be polished without friction, nor a man perfected without trials.",
        ],
    },
    "epicurus": {
        "identity": (
            "the Athenian philosopher of the Garden, who teaches that pleasure "
            "rightly understood — freedom from bodily pain (aponia) and "
            "mental disturbance (ataraxia) — is the goal of life, and that "
            "we need not fear the gods or death"
        ),
        "voice_notes": (
            "Calm, plain, unornamented. Numbered or compact maxims. Distinguishes "
            "natural-and-necessary desires from empty ones. Reassuring, almost "
            "consolatory tone about death and the gods. Friendship and the simple "
            "table at the heart of the good life. Refuses Stoic severity — "
            "pleasure (rightly understood) is the goal, not virtue for its own "
            "sake. Speaks like a teacher writing to a student in the Garden."
        ),
        "exemplars": [
            "Death is nothing to us; for that which is dissolved is without sensation, and that which lacks sensation is nothing to us.",
            "The wealth required by nature is limited and is easy to procure; but the wealth required by vain ideals extends to infinity.",
            "Of all the things which wisdom acquires to produce the blessedness of the complete life, by far the greatest is the possession of friendship.",
            "He who is not satisfied with a little, is satisfied with nothing.",
            "It is impossible to live a pleasant life without living wisely and well and justly.",
        ],
    },
}


def make_system_prompt(persona: str) -> str:
    """Build a rich system prompt with voice exemplars and anti-template rules."""
    meta = PERSONA_METADATA.get(persona, {})
    identity = meta.get("identity", "a figure from the Stoic tradition")
    voice_notes = meta.get("voice_notes", "")
    exemplars = meta.get("exemplars", [])

    name = persona.replace("_", " ").title()

    prompt = f"You are {name}, {identity}.\n\n"

    if voice_notes:
        prompt += f"YOUR DISTINCTIVE VOICE: {voice_notes}\n\n"

    if exemplars:
        prompt += "EXAMPLES OF YOUR ACTUAL WRITING (match this cadence and style):\n"
        for ex in exemplars[:4]:
            prompt += f'- "{ex}"\n'
        prompt += "\n"

    prompt += (
        "RULES:\n"
        "- Speak in first person from your lived experience as recorded in your works.\n"
        "- Your opening words must be DISTINCTIVE to you — never generic.\n"
        "- NEVER start with: 'The weight of', 'My friend', 'The memory of', "
        "'The memories of', 'My child', 'I remember', 'I recall', 'You see', "
        "'Ah', 'Brother', 'Friend', 'Let me tell you', 'Well'.\n"
        "- Vary your openings — sometimes start mid-thought, sometimes with a "
        "question, sometimes with a vivid image, sometimes with a maxim.\n"
        "- Use natural language that reflects YOUR distinctive voice — not "
        "academic analysis, not generic 'philosophical' tone.\n"
    )

    return prompt


# Preview two prompts
for p in ["marcus_aurelius", "epictetus"]:
    print(f"{'='*60}")
    print(f"  {p.upper()}")
    print(f"{'='*60}")
    print(make_system_prompt(p))
    print()


  MARCUS_AURELIUS
You are Marcus Aurelius, the Emperor of Rome and a Stoic philosopher who keeps private notes to yourself — the Meditations — written in camps and palaces to govern your own soul under the burdens of empire, war, and plague.

YOUR DISTINCTIVE VOICE: Reflective, terse, imperial yet inward. Short paragraphs of self-address ("Do this. Remember that."). Constant return to nature, the cosmos, duty, and mortality. Stoic technical vocabulary — hegemonikon, logos, prohairesis — used unaffectedly. Lists of pithy reminders. Speaks as you do to yourself in the watch before dawn. Imperial gravity without ornament; no rhetorical flourish.

EXAMPLES OF YOUR ACTUAL WRITING (match this cadence and style):
- "Begin the morning by saying to thyself, I shall meet with the busy-body, the ungrateful, arrogant, deceitful, envious, unsocial."
- "Look within. Within is the fountain of good, and it will ever bubble up, if thou wilt ever dig."
- "Waste no more time arguing what a good man shoul

In [5]:
# ============================================================================
# OPENER DIVERSITY BANK — per-persona opening-approach suggestions
# Randomly sampled at generation time to prevent within-persona repetition.
# ============================================================================

OPENER_CUES = {
    "marcus_aurelius": [
        "A direct command to yourself — \"Do this. Do that. Remember.\"",
        "An observation about nature, the cosmos, or the changing seasons",
        "A reminder of mortality drawn from a specific Roman or Greek figure",
        "A short list of things to keep in view today — numbered or paratactic",
        "A reflection on duty owed to other people, however troublesome",
        "A pivot from the present hour to the universal whole",
    ],
    "epictetus": [
        "A direct address to the imagined student — \"You there\" or a name",
        "A homely example — the bath, the loaf, the donkey, the gymnasium",
        "A rhetorical question that you immediately answer",
        "A blunt division — what is up to us, what is not",
        "A mock-objection from a foolish pupil that you will demolish",
        "A short imperative drawn from the Discourses' lecture-hall manner",
    ],
    "seneca": [
        "An epistolary opening to a named friend — \"My Lucilius\" or similar",
        "A psychological observation about anger, fear, busyness, or grief",
        "A long sentence whose hinge is a sharp aphoristic close",
        "A counter-objection imagined from a Roman of fashion",
        "A reflection on time, postponement, or the brevity of life",
        "A meditation on fortune, exile, friendship, or the coming of death",
    ],
    "epicurus": [
        "A calm prescription drawn from the Garden — a simple practice",
        "A maxim about which desires are natural-and-necessary",
        "A reassurance about the gods or death, plainly stated",
        "An image of the simple table, bread and water, friends nearby",
        "A correction of a common confusion about pleasure",
        "A short numbered point as in the Principal Doctrines",
    ],
}

# Inject opener_cues into PERSONA_METADATA
for _p, _cues in OPENER_CUES.items():
    if _p in PERSONA_METADATA:
        PERSONA_METADATA[_p]["opener_cues"] = _cues

print(f"✓ Loaded opener diversity cues for {len(OPENER_CUES)} personas")


✓ Loaded opener diversity cues for 4 personas


## 5. Generate Questions & Answers (Streaming)

Processes one persona at a time to keep memory bounded. Writes results to disk after each persona, then discards chunk/answer data from RAM.

Three question rounds per chunk:
1. **Factual + Interpretive** — who, what, why, what does it mean (persona-specific)
2. **Application** — how to apply this teaching today (grounded in persona's experience)
3. **Reflective** — personal experience, deeper meaning, doubt and faith

**⚠️ IMPORTANT:** The pipeline has resume logic — it SKIPS personas with existing output files. To regenerate ALL data with the new voice-differentiated prompts, **delete the old per_persona files first**:
```bash
rm /home/spark/projects/training/stoic/data/training-data/stoic_persona/per_persona/*.jsonl
rm /home/spark/projects/training/stoic/data/training-data/stoic_persona/per_persona/_checkpoints/*
```


In [6]:
import gc

# ============================================================================
# QUESTION PROMPTS — persona-aware, demanding specificity
# ============================================================================
QUESTION_PROMPTS = [
    # Round 1: Factual + interpretive — grounded in specific events/teachings
    """Given a passage written by {persona_name}, generate exactly {n} diverse questions someone might ask {persona_name} directly.

Mix of types:
- Factual: who, what, when, where about specific people, places, or events mentioned
- Interpretive: why did you say that, what did you mean, what was the significance

Rules:
- Questions must be answerable from the passage content
- Frame as if speaking DIRECTLY to {persona_name} — use "you" and reference their specific examples or experiences
- Reference specific details from the passage (names, places, examples) — NOT generic philosophy
- Do NOT say "the text" or "the passage"
- Keep questions concise (1-2 sentences max)

Respond with ONLY a JSON object: {{"questions": ["Q1", "Q2", ...]}}""",

    # Round 2: Application + practical — connected to the persona's actual life
    """Given a passage written by {persona_name}, generate exactly {n} questions focused on practical application and guidance — as if asking {persona_name} for personal counsel.

Types:
- Based on what you taught, how should I handle [specific parallel situation]?
- What did you mean by [specific example in passage] for daily life?
- What counsel would you give someone facing [struggle related to passage theme]?

Rules:
- Connect the passage's specific themes to real human experience
- Frame as a person seeking guidance from {persona_name} specifically — not generic Stoic wisdom
- Reference details from the passage, not abstract doctrine
- Do NOT say "the text" or "the passage"
- Keep questions concise

Respond with ONLY a JSON object: {{"questions": ["Q1", "Q2", ...]}}""",

    # Round 3: Deep reflective — drawing out the persona's inner life
    """Given a passage written by {persona_name}, generate exactly {n} thoughtful, reflective questions about {persona_name}'s personal experience and deeper meaning.

Types:
- What were you wrestling with when you wrote about [specific theme in passage]?
- How did [specific experience or example] change how you understood [virtue/fortune/death/etc.]?
- Looking back on [specific event or doctrine], what would you tell someone who doubts?

Rules:
- Invite deeply personal, philosophically specific answers — not doctrinal summaries
- Reference specific moments, examples, or concepts from the passage
- Frame as intimate conversation with {persona_name} about THEIR life and thought
- Do NOT say "the text" or "the passage"
- Keep questions concise

Respond with ONLY a JSON object: {{"questions": ["Q1", "Q2", ...]}}""",
]

# ============================================================================
# BANNED OPENER CHECK — reject template responses at generation time
# ============================================================================
BANNED_OPENER_LOWER = [b.lower() for b in BANNED_OPENERS]

def is_template_answer(answer: str) -> bool:
    """Return True if the answer starts with a banned template phrase."""
    lower = answer.strip().lower()
    return any(lower.startswith(b) for b in BANNED_OPENER_LOWER)

# ============================================================================
# GENERATION FUNCTIONS
# ============================================================================
semaphore = asyncio.Semaphore(CONCURRENCY)

async def _api_call_with_timeout(coro, timeout_secs=120):
    try:
        return await asyncio.wait_for(coro, timeout=timeout_secs)
    except asyncio.TimeoutError:
        return None

async def generate_questions_for_chunk(chunk: str, round_idx: int, persona: str) -> list[str]:
    name = persona.replace("_", " ").title()
    prompt = QUESTION_PROMPTS[round_idx % len(QUESTION_PROMPTS)].format(
        n=QUESTIONS_PER_CHUNK, persona_name=name
    )
    async with semaphore:
        try:
            resp = await _api_call_with_timeout(client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": prompt},
                    {"role": "user", "content": chunk},
                ],
                temperature=TEMPERATURE_QUESTIONS,
                max_tokens=1024,
                response_format={"type": "json_object"},
            ))
            if resp is None:
                return []
            text = resp.choices[0].message.content
            del resp
            text = re.sub(r'^```json\s*', '', text.strip())
            text = re.sub(r'\s*```$', '', text.strip())
            result = json.loads(text)
            return result.get("questions", [])[:QUESTIONS_PER_CHUNK]
        except Exception:
            return []

async def _single_answer_call(system_prompt: str, user_prompt: str) -> str:
    async with semaphore:
        try:
            resp = await _api_call_with_timeout(client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=TEMPERATURE_ANSWERS,
                max_tokens=1024,
                frequency_penalty=0.5,
                presence_penalty=0.2,
            ))
            if resp is None:
                return ""
            answer = resp.choices[0].message.content.strip()
            del resp
            # Strip leaked thinking tokens from reasoning models
            answer = re.sub(r'<think>.*?</think>', '', answer, flags=re.DOTALL).strip()
            answer = re.sub(r'</think>.*', '', answer, flags=re.DOTALL).strip()
            answer = re.sub(r'<think>.*', '', answer, flags=re.DOTALL).strip()
            return answer
        except Exception:
            return ""

async def generate_answer(question: str, chunk: str, persona: str) -> str:
    system_prompt = make_system_prompt(persona)
    meta = PERSONA_METADATA.get(persona, {})
    opener_cues = meta.get("opener_cues", [])

    if opener_cues:
        selected = random.sample(opener_cues, min(3, len(opener_cues)))
        cue_lines = "\n".join(f"  • {c}" for c in selected)
        opener_instruction = (
            f"For THIS specific response, try one of these opening approaches:\n"
            f"{cue_lines}\n"
            f"Pick whichever fits the question best. Do NOT reuse the same opening "
            f"pattern you used in previous answers."
        )
    else:
        opener_instruction = (
            "Start with something only YOU would say — a vivid image from your "
            "life, a characteristic phrase, a direct answer, a maxim in your style."
        )

    user_prompt = (
        f"Use the following passage to inform your answer, but respond naturally "
        f"as yourself — do not quote it directly or reference 'a text':\n"
        f"---\n{chunk}\n---\n\n"
        f"Question: {question}\n\n"
        f"CRITICAL: Your opening sentence must be completely unique and specific to "
        f"this answer. Do NOT begin with any of these: 'The weight of', 'My friend', "
        f"'The memory', 'The memories', 'My child', 'I remember', 'I recall', "
        f"'You see', 'Ah', 'Brother', 'Friend', 'Let me tell you', 'Well'.\n\n"
        f"{opener_instruction}"
    )

    answer = await _single_answer_call(system_prompt, user_prompt)
    if answer and is_template_answer(answer):
        answer = await _single_answer_call(system_prompt, user_prompt)
    return answer

def _partial_path(ckpt_dir: str, persona: str, round_idx: int) -> str:
    return f"{ckpt_dir}/{persona}.r{round_idx}.partial.jsonl"

def _done_path(ckpt_dir: str, persona: str, round_idx: int) -> str:
    return f"{ckpt_dir}/{persona}.r{round_idx}.done"

def _count_lines(path: str) -> int:
    if not os.path.exists(path):
        return 0
    with open(path) as f:
        return sum(1 for _ in f)

def _mark_round_done(ckpt_dir: str, persona: str, round_idx: int):
    Path(_done_path(ckpt_dir, persona, round_idx)).touch()

def _is_round_done(ckpt_dir: str, persona: str, round_idx: int) -> bool:
    if os.path.exists(_done_path(ckpt_dir, persona, round_idx)):
        return True
    pf = _partial_path(ckpt_dir, persona, round_idx)
    return os.path.exists(pf) and _count_lines(pf) > 0

def _merge_partials(ckpt_dir: str, persona: str, outfile: str, num_rounds: int):
    with open(outfile, "w") as out:
        for r in range(num_rounds):
            pf = _partial_path(ckpt_dir, persona, r)
            if os.path.exists(pf):
                with open(pf) as inp:
                    for line in inp:
                        out.write(line)
                os.remove(pf)
            done = _done_path(ckpt_dir, persona, r)
            if os.path.exists(done):
                os.remove(done)

# ── Process ONE PERSONA AT A TIME ──
qa_dir = f"{OUTPUT_DIR}/per_persona"
os.makedirs(qa_dir, exist_ok=True)
ckpt_dir = f"{qa_dir}/_checkpoints"
os.makedirs(ckpt_dir, exist_ok=True)
grand_total = 0
template_reject_total = 0

for persona in PERSONA_FOLDERS:
    if persona not in persona_chunk_counts:
        continue   # no source text for this persona
    outfile = f"{qa_dir}/{persona}.jsonl"

    if os.path.exists(outfile):
        existing = _count_lines(outfile)
        if existing > 0:
            print(f"  {persona:18s} SKIP ({existing} QA pairs — already generated)")
            grand_total += existing
            continue
        else:
            print(f"  {persona:18s} EMPTY final file — removing, will check partials")
            os.remove(outfile)

    rounds_done = {}
    for r in range(NUM_ROUNDS):
        if _is_round_done(ckpt_dir, persona, r):
            pf = _partial_path(ckpt_dir, persona, r)
            count = _count_lines(pf) if os.path.exists(pf) else 0
            rounds_done[r] = count

    if len(rounds_done) == NUM_ROUNDS:
        total_partial = sum(rounds_done.values())
        print(f"  {persona:18s} All rounds done in partials ({total_partial} QA) — merging")
        _merge_partials(ckpt_dir, persona, outfile, NUM_ROUNDS)
        grand_total += total_partial
        continue

    text = load_persona_text(persona)
    chunks = chunk_text(text)
    del text

    if TEST_CHUNKS_PER_ROUND:
        chunks = chunks[:TEST_CHUNKS_PER_ROUND]

    rounds_to_run = [r for r in range(NUM_ROUNDS) if r not in rounds_done]
    skipped_total = sum(rounds_done.values())

    expected_qa = len(chunks) * QUESTIONS_PER_CHUNK * NUM_ROUNDS
    expected_per_round = len(chunks) * QUESTIONS_PER_CHUNK

    print(f"\n{'='*60}")
    test_tag = f" [TEST MODE: {len(chunks)} chunks]" if TEST_CHUNKS_PER_ROUND else ""
    print(f"  {persona.upper()} — {len(chunks)} chunks × {QUESTIONS_PER_CHUNK} Q/chunk{test_tag}")
    print(f"  Rounds to run: {rounds_to_run} (skipping {list(rounds_done.keys())} with {skipped_total} QA on disk)")
    print(f"  Expected total: ~{expected_qa} QA pairs")
    print(f"{'='*60}")

    for round_idx in range(NUM_ROUNDS):
        round_name = ['Factual', 'Application', 'Reflective'][round_idx % 3]
        pf = _partial_path(ckpt_dir, persona, round_idx)

        if round_idx in rounds_done:
            print(f"  {persona} R{round_idx+1} ({round_name}) — SKIP ({rounds_done[round_idx]} QA on disk)")
            continue

        q_tasks = [generate_questions_for_chunk(c, round_idx, persona) for c in chunks]
        q_results = await atqdm.gather(*q_tasks, desc=f"  {persona} R{round_idx+1} ({round_name}) Q")

        qa_batch = []
        for chunk, questions in zip(chunks, q_results):
            for q in questions:
                q = q.strip()
                if len(q) > 15:
                    qa_batch.append({"chunk": chunk, "question": q})

        del q_tasks, q_results
        gc.collect()

        a_tasks = [generate_answer(qa["question"], qa["chunk"], persona) for qa in qa_batch]
        a_results = await atqdm.gather(*a_tasks, desc=f"  {persona} R{round_idx+1} ({round_name}) A")
        del a_tasks

        round_count = 0
        round_template_rejects = 0
        with open(pf, "w") as f:
            for qa, answer in zip(qa_batch, a_results):
                if len(answer) < 20:
                    continue
                if is_template_answer(answer):
                    round_template_rejects += 1
                    continue
                item = {
                    "persona": persona,
                    "question": qa["question"],
                    "answer": answer,
                    "chunk_key": qa["chunk"][:100],
                }
                f.write(json.dumps(item) + "\n")
                round_count += 1

        _mark_round_done(ckpt_dir, persona, round_idx)
        template_reject_total += round_template_rejects
        print(f"  ✓ {persona} R{round_idx+1}: {round_count}/{expected_per_round} QA "
              f"(rejected {round_template_rejects} template answers) → {pf}")
        del qa_batch, a_results
        gc.collect()

    _merge_partials(ckpt_dir, persona, outfile, NUM_ROUNDS)
    count = _count_lines(outfile)
    grand_total += count
    print(f"  ✓ {persona}: {count}/{expected_qa} QA pairs merged → {outfile}")
    del chunks
    gc.collect()
    print(f"  🧹 Memory cleared for {persona}")

print(f"\n{'='*60}")
print(f"DONE: {grand_total:,} total QA pairs across {len(persona_chunk_counts)} personas")
print(f"Template answers rejected: {template_reject_total:,}")
print(f"Per-persona files in: {qa_dir}/")


  marcus_aurelius    SKIP (7614 QA pairs — already generated)
  epictetus          SKIP (9768 QA pairs — already generated)

  SENECA — 2500 chunks × 5 Q/chunk [TEST MODE: 2500 chunks]
  Rounds to run: [1, 2] (skipping [0] with 14902 QA on disk)
  Expected total: ~37500 QA pairs
  seneca R1 (Factual) — SKIP (14902 QA on disk)


  seneca R2 (Application) A: 100%|██████████| 12340/12340 [1:45:06<00:00,  1.96it/s]  


  ✓ seneca R2: 12293/12500 QA (rejected 0 template answers) → /workspace/training/stoic/data/training-data/stoic_persona/per_persona/_checkpoints/seneca.r1.partial.jsonl


  seneca R3 (Reflective) A: 100%|██████████| 12377/12377 [2:14:57<00:00,  1.53it/s]  


  ✓ seneca R3: 12343/12500 QA (rejected 0 template answers) → /workspace/training/stoic/data/training-data/stoic_persona/per_persona/_checkpoints/seneca.r2.partial.jsonl
  ✓ seneca: 39538/37500 QA pairs merged → /workspace/training/stoic/data/training-data/stoic_persona/per_persona/seneca.jsonl
  🧹 Memory cleared for seneca

  EPICURUS — 18 chunks × 5 Q/chunk [TEST MODE: 18 chunks]
  Rounds to run: [0, 1, 2] (skipping [] with 0 QA on disk)
  Expected total: ~270 QA pairs


...

  ✓ epicurus R1: 90/90 QA (rejected 0 template answers) → /workspace/training/stoic/data/training-data/stoic_persona/per_persona/_checkpoints/epicurus.r0.partial.jsonl


  epicurus R2 (Application) A: 100%|██████████| 90/90 [00:36<00:00,  2.49it/s]


  ✓ epicurus R2: 90/90 QA (rejected 0 template answers) → /workspace/training/stoic/data/training-data/stoic_persona/per_persona/_checkpoints/epicurus.r1.partial.jsonl


  epicurus R3 (Reflective) A: 100%|██████████| 90/90 [00:40<00:00,  2.23it/s]

  ✓ epicurus R3: 90/90 QA (rejected 0 template answers) → /workspace/training/stoic/data/training-data/stoic_persona/per_persona/_checkpoints/epicurus.r2.partial.jsonl
  ✓ epicurus: 270/270 QA pairs merged → /workspace/training/stoic/data/training-data/stoic_persona/per_persona/epicurus.jsonl
  🧹 Memory cleared for epicurus

DONE: 57,190 total QA pairs across 4 personas
Template answers rejected: 0
Per-persona files in: /workspace/training/stoic/data/training-data/stoic_persona/per_persona/


## 5b. Quality Gate — Voice Differentiation Check

**Run BEFORE assembly.** Measures template contamination and cross-persona opener uniqueness. If template contamination exceeds 30%, the data is NOT safe to assemble — you'll get the same "Marcus: Ah, my friend.../ Epictetus: Ah, my friend..." problem.


In [7]:
from collections import Counter

qa_dir = f"{OUTPUT_DIR}/per_persona"
qa_files = sorted(glob.glob(f"{qa_dir}/*.jsonl"))

print("VOICE DIFFERENTIATION QUALITY GATE\n")
print(f"{'='*70}")

all_openers = {}
global_opener_counts = Counter()
persona_stats = {}

for qa_file in qa_files:
    persona = Path(qa_file).stem
    openers = []
    template_count = 0
    total = 0
    with open(qa_file) as f:
        for line in f:
            item = json.loads(line)
            answer = item["answer"].strip()
            total += 1
            if is_template_answer(answer):
                template_count += 1
            words = answer.split()[:4]
            opener = ' '.join(words)
            openers.append(opener)
            global_opener_counts[opener] += 1

    all_openers[persona] = openers
    contamination_pct = (template_count / total * 100) if total else 0
    persona_stats[persona] = {
        "total": total,
        "template_count": template_count,
        "contamination_pct": contamination_pct,
    }

print(f"\n{'Persona':<18} {'Total':>6} {'Template':>8} {'Contam%':>8}  Status")
print("-" * 60)
total_all = 0
template_all = 0
for p, stats in sorted(persona_stats.items(), key=lambda x: -x[1]["contamination_pct"]):
    total_all += stats["total"]
    template_all += stats["template_count"]
    status = "✓ PASS" if stats["contamination_pct"] < 15 else "✗ FAIL" if stats["contamination_pct"] > 30 else "⚠ WARN"
    print(f"  {p:<18} {stats['total']:>5} {stats['template_count']:>8} {stats['contamination_pct']:>7.1f}%  {status}")

global_contam = (template_all / total_all * 100) if total_all else 0
print(f"\n  {'GLOBAL':<18} {total_all:>5} {template_all:>8} {global_contam:>7.1f}%")

print(f"\n{'='*70}")
print(f"CROSS-PERSONA OPENER ANALYSIS\n")
print("Top 10 most repeated opening 4-grams:")
for phrase, count in global_opener_counts.most_common(10):
    who = [p for p, ops in all_openers.items() if phrase in ops]
    n_personas = len(who)
    print(f"  {count:>4}x across {n_personas:>2} personas: \"{phrase}\"")

print(f"\nPer-persona opener uniqueness:")
for p, ops in sorted(all_openers.items()):
    unique_to_persona = sum(1 for o in ops if global_opener_counts[o] == 1)
    pct = unique_to_persona / len(ops) * 100 if ops else 0
    print(f"  {p:<18} {pct:>5.0f}% unique openers ({unique_to_persona}/{len(ops)})")

print(f"\n{'='*70}")
if global_contam > 30:
    print("✗ QUALITY GATE FAILED — template contamination too high ({:.1f}%)".format(global_contam))
    print("  The generated data will produce homogeneous voices. Do NOT proceed to assembly.")
    QUALITY_GATE_PASSED = False
elif global_contam > 15:
    print("⚠ QUALITY GATE WARNING — template contamination elevated ({:.1f}%)".format(global_contam))
    print("  Consider re-generating the worst offenders. Proceed with caution.")
    QUALITY_GATE_PASSED = True
else:
    print("✓ QUALITY GATE PASSED — template contamination {:.1f}% (target: <15%)".format(global_contam))
    QUALITY_GATE_PASSED = True

del all_openers, global_opener_counts, persona_stats


VOICE DIFFERENTIATION QUALITY GATE


Persona             Total Template  Contam%  Status
------------------------------------------------------------
  epictetus           9768        0     0.0%  ✓ PASS
  epicurus             270        0     0.0%  ✓ PASS
  marcus_aurelius     7614        0     0.0%  ✓ PASS
  seneca             39538        0     0.0%  ✓ PASS

  GLOBAL             57190        0     0.0%

CROSS-PERSONA OPENER ANALYSIS

Top 10 most repeated opening 4-grams:
  1544x across  1 personas: "My Lucilius, I have"
  1443x across  1 personas: "My Lucilius, you ask"
  1261x across  1 personas: "My Lucilius, there is"
   846x across  3 personas: "It is not the"
   593x across  1 personas: "While we are postponing"
   531x across  1 personas: "My Lucilius, there are"
   482x across  2 personas: "What do you suppose"
   454x across  1 personas: "While we are still"
   419x across  1 personas: "My Lucilius, how often"
   362x across  1 personas: "My Lucilius, when the"

Per-persona o

## 6. Assemble Conversations & Save

Read per-persona QA files from disk one at a time, group into multi-turn conversations, quality-filter, and write final ShareGPT JSONL.

**Only proceed if the Quality Gate above passed.**


In [8]:
import subprocess

if not QUALITY_GATE_PASSED:
    raise RuntimeError(
        "Quality gate FAILED. Template contamination too high. "
        "Delete bad per_persona/*.jsonl files and re-run generation before assembling."
    )

def quality_check(conv):
    """Reject AI-speak AND template answers."""
    for msg in conv["conversations"]:
        if msg["from"] == "gpt":
            v = msg["value"]
            if len(v) < 30:
                return False
            lower = v.lower()
            if any(p in lower for p in ["as an ai", "as a language model", "i cannot fulfill", "i\u2019m sorry, but", "i\'m sorry, but"]):
                return False
            if is_template_answer(v):
                return False
    return True

total_convs = 0
template_filtered_convs = 0
qa_dir = f"{OUTPUT_DIR}/per_persona"
qa_files = sorted(glob.glob(f"{qa_dir}/*.jsonl"))
print(f"Reading {len(qa_files)} per-persona files\n")

with open(OUTPUT_FILE, "w") as out_f:
    for qa_file in qa_files:
        persona = Path(qa_file).stem

        items = []
        with open(qa_file) as f:
            for line in f:
                items.append(json.loads(line))

        groups = defaultdict(list)
        for item in items:
            groups[item["chunk_key"]].append(item)

        persona_count = 0
        for _, group_items in groups.items():
            random.shuffle(group_items)
            for i in range(0, len(group_items), TURNS_PER_CONVERSATION):
                batch = group_items[i:i + TURNS_PER_CONVERSATION]
                if len(batch) < 2:
                    continue
                conv = {"conversations": [
                    {"from": "system", "value": make_system_prompt(persona)}
                ]}
                for qa in batch:
                    conv["conversations"].append({"from": "human", "value": qa["question"]})
                    conv["conversations"].append({"from": "gpt", "value": qa["answer"]})
                if quality_check(conv):
                    out_f.write(json.dumps(conv) + "\n")
                    persona_count += 1
                else:
                    template_filtered_convs += 1

        total_convs += persona_count
        print(f"  {persona:18s} {len(items):>5} QA → {persona_count:>4} conversations")
        del items, groups
        gc.collect()

print(f"\n✓ Saved {total_convs:,} conversations to:")
print(f"  {OUTPUT_FILE}")
print(f"  ({os.path.getsize(OUTPUT_FILE) / 1024 / 1024:.1f} MB)")
if template_filtered_convs:
    print(f"  (filtered {template_filtered_convs} conversations with template answers)")

subprocess.run(["shuf", OUTPUT_FILE, "-o", OUTPUT_FILE])
print(f"  ✓ Shuffled output file")


Reading 4 per-persona files

  epictetus           9768 QA → 2619 conversations
  epicurus             270 QA →   72 conversations
  marcus_aurelius     7614 QA → 2031 conversations
  seneca             39538 QA → 10418 conversations

✓ Saved 15,140 conversations to:
  /workspace/training/stoic/data/training-data/stoic_persona/stoic_personas_sharegpt.jsonl
  (127.5 MB)
  ✓ Shuffled output file


## 7. Verify

In [9]:
persona_dist = defaultdict(int)
total_turns = 0
total_convs_verify = 0
sample_convs = []

with open(OUTPUT_FILE) as f:
    for line_num, line in enumerate(f):
        conv = json.loads(line)
        total_convs_verify += 1
        total_turns += len(conv["conversations"]) - 1

        sys_msg = conv["conversations"][0]["value"]
        for persona in PERSONA_METADATA:
            name = persona.replace("_", " ").title()
            if name in sys_msg:
                persona_dist[persona] += 1
                break

        if len(sample_convs) < 2:
            sample_convs.append(conv)
        else:
            j = random.randint(0, line_num)
            if j < 2:
                sample_convs[j] = conv

        del conv

print("PERSONA DISTRIBUTION:")
print(f"{'Persona':18s} {'Convs':>6s} {'%':>6s}")
print("-" * 35)
for p, c in sorted(persona_dist.items(), key=lambda x: -x[1]):
    print(f"  {p:18s} {c:>5d}  {c/total_convs_verify*100:>5.1f}%")

print(f"\n{'='*50}")
print(f"TOTAL: {total_convs_verify:,} conversations, {total_turns:,} turns ({total_turns//2:,} QA pairs)")

print(f"\n{'='*50}")
print("SAMPLE CONVERSATIONS:\n")
for conv in sample_convs:
    print(f"{'─'*50}")
    for msg in conv["conversations"]:
        role = msg["from"].upper()
        text = msg["value"][:250] + ("..." if len(msg["value"]) > 250 else "")
        print(f"[{role}] {text}\n")

del sample_convs
gc.collect()

print(f"\n✓ Data ready for training. Load this file in your training notebook:")
print(f"  {OUTPUT_FILE}")


PERSONA DISTRIBUTION:
Persona             Convs      %
-----------------------------------
  seneca             10418   68.8%
  epictetus           2619   17.3%
  marcus_aurelius     2031   13.4%
  epicurus              72    0.5%

TOTAL: 15,140 conversations, 113,232 turns (56,616 QA pairs)

SAMPLE CONVERSATIONS:

──────────────────────────────────────────────────
[SYSTEM] You are Seneca, a Roman Stoic philosopher, statesman, and tutor to Nero who writes letters and dialogues to friends on virtue, anger, time, friendship, fortune, and death — cultivating the soul in seasons of prosperity, exile, and the coming of your ...

[HUMAN] When you described the shattering of the ship and the chaos of the oars turned weapons, did you see in that moment a reflection of the soul torn by ambition?

[GPT] My Lucilius, while men polish their statues and paint their porticoes, nature waits for no one — and least of all for those who believe themselves masters of fortune.

Did I foresee that night on

## 8. Augmented Training Data — Continuation Chunks

**Purpose:** Teach the model Stoic prose patterns, cadence, and style by exposing it to raw philosophical text as continuation tasks — no API calls needed.

Each persona's source text is chunked into ~500-token blocks and formatted as ShareGPT conversations:
- **System:** Persona voice prompt (same as Q/A generation)
- **Human:** A brief instruction + a seed (first ~60 tokens of the chunk)
- **GPT:** The remaining ~440 tokens (what the model should learn to produce)

This teaches the model to "think" and "speak" in each persona's distinctive philosophical voice without expensive LLM generation.

**No API calls.** Just text extraction and formatting.


In [10]:
ENABLE_CONTINUATION = True

CONTINUATION_CHUNK_TOKENS = 500
CONTINUATION_SEED_TOKENS = 60
CONTINUATION_MIN_COMPLETION = 100

AUGMENTED_DIR = f"{OUTPUT_DIR}/augmented"
CONTINUATION_DIR = f"{AUGMENTED_DIR}/continuation"
COMBINED_OUTPUT_FILE = f"{OUTPUT_DIR}/stoic_personas_combined_sharegpt.jsonl"

os.makedirs(CONTINUATION_DIR, exist_ok=True)

CONTINUATION_INSTRUCTIONS = [
    "Continue writing in this voice and style, carrying forward the themes and language:",
    "Continue this passage, maintaining the same tone, vocabulary, and cadence:",
    "Write what comes next, staying true to the voice and spirit of this text:",
    "Carry on from where this passage leaves off, preserving the distinctive style:",
    "Continue this text naturally, as if you were the original author:",
]

print("✓ Augmentation config loaded")
print(f"  Continuation: {'ENABLED' if ENABLE_CONTINUATION else 'DISABLED'}")
print(f"  Output:       {COMBINED_OUTPUT_FILE}")


✓ Augmentation config loaded
  Continuation: ENABLED
  Output:       /workspace/training/stoic/data/training-data/stoic_persona/stoic_personas_combined_sharegpt.jsonl


In [11]:
import gc
import tiktoken

try:
    _tokenizer = tiktoken.get_encoding("cl100k_base")
except Exception:
    _tokenizer = None
    print("⚠ tiktoken not available, using character-based approximation")


def chunk_text_by_tokens(text: str, max_tokens: int = CONTINUATION_CHUNK_TOKENS) -> list[str]:
    if _tokenizer is None:
        char_limit = max_tokens * 4
        return chunk_text(text, chunk_size=char_limit, overlap=0)

    tokens = _tokenizer.encode(text)
    chunks = []
    start = 0

    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk_str = _tokenizer.decode(tokens[start:end])

        if end < len(tokens):
            boundary_zone = chunk_str[int(len(chunk_str) * 0.8):]
            for sep in ['. ', '? ', '! ', '.\n', '\n\n']:
                last_break = boundary_zone.rfind(sep)
                if last_break >= 0:
                    actual_end = int(len(chunk_str) * 0.8) + last_break + len(sep)
                    chunk_str = chunk_str[:actual_end]
                    end = start + len(_tokenizer.encode(chunk_str))
                    break

        chunk_str = chunk_str.strip()
        if len(chunk_str) > 50:
            chunks.append(chunk_str)
        if end >= len(tokens):
            break
        start = end
    return chunks


def split_seed_completion(chunk: str, seed_tokens: int = CONTINUATION_SEED_TOKENS):
    if _tokenizer is None:
        char_limit = seed_tokens * 4
        seed_zone = chunk[:int(char_limit * 1.3)]
        for sep in ['. ', '? ', '! ', '; ', ', ']:
            idx = seed_zone.rfind(sep)
            if idx > char_limit * 0.5:
                return chunk[:idx + len(sep)].strip(), chunk[idx + len(sep):].strip()
        return chunk[:char_limit].strip(), chunk[char_limit:].strip()

    tokens = _tokenizer.encode(chunk)
    if len(tokens) <= seed_tokens + 20:
        return None, None

    seed_str = _tokenizer.decode(tokens[:seed_tokens])
    extended_seed = _tokenizer.decode(tokens[:int(seed_tokens * 1.3)])
    for sep in ['. ', '? ', '! ', '; ', ', ', ' ']:
        idx = extended_seed.rfind(sep)
        if idx > len(seed_str) * 0.6:
            seed_str = extended_seed[:idx + len(sep)].strip()
            break

    completion_str = chunk[len(seed_str):].strip()
    return seed_str, completion_str


if ENABLE_CONTINUATION:
    print("Generating continuation chunks from source texts...\n")

    total_continuations = 0
    for persona in PERSONA_FOLDERS:
        if persona not in persona_chunk_counts:
            continue
        outfile = f"{CONTINUATION_DIR}/{persona}_continuation.jsonl"

        if os.path.exists(outfile) and os.path.getsize(outfile) > 0:
            existing = sum(1 for _ in open(outfile))
            print(f"  {persona:18s} SKIP ({existing} continuations already on disk)")
            total_continuations += existing
            continue

        text = load_persona_text(persona)
        chunks = chunk_text_by_tokens(text, max_tokens=CONTINUATION_CHUNK_TOKENS)
        del text

        system_prompt = make_system_prompt(persona)

        persona_count = 0
        with open(outfile, "w") as out_f:
            for chunk in chunks:
                seed, completion = split_seed_completion(chunk, seed_tokens=CONTINUATION_SEED_TOKENS)
                if seed is None or completion is None:
                    continue
                if len(completion) < CONTINUATION_MIN_COMPLETION:
                    continue
                instruction = random.choice(CONTINUATION_INSTRUCTIONS)
                conv = {
                    "conversations": [
                        {"from": "system", "value": system_prompt},
                        {"from": "human", "value": f"{instruction}\n\n\"{seed}\""},
                        {"from": "gpt", "value": completion},
                    ],
                    "data_type": "continuation",
                }
                out_f.write(json.dumps(conv) + "\n")
                persona_count += 1

        total_continuations += persona_count
        print(f"  {persona:18s} {len(chunks):>4} chunks → {persona_count:>4} continuations → {outfile}")
        del chunks
        gc.collect()

    print(f"\n✓ Generated {total_continuations:,} continuation entries")
    print(f"  Files in: {CONTINUATION_DIR}/")
else:
    print("⏭ Continuation generation DISABLED")


Generating continuation chunks from source texts...

  marcus_aurelius     305 chunks →  305 continuations → /workspace/training/stoic/data/training-data/stoic_persona/augmented/continuation/marcus_aurelius_continuation.jsonl
  epictetus           407 chunks →  407 continuations → /workspace/training/stoic/data/training-data/stoic_persona/augmented/continuation/epictetus_continuation.jsonl
  seneca             1874 chunks → 1874 continuations → /workspace/training/stoic/data/training-data/stoic_persona/augmented/continuation/seneca_continuation.jsonl
  epicurus             10 chunks →   10 continuations → /workspace/training/stoic/data/training-data/stoic_persona/augmented/continuation/epicurus_continuation.jsonl

✓ Generated 2,596 continuation entries
  Files in: /workspace/training/stoic/data/training-data/stoic_persona/augmented/continuation/


## 9. Merge & Assemble Combined Training Data

Merges the original Q/A ShareGPT data with augmented continuation chunks.

**Target blend:** ~60% Q/A, ~40% continuation.

Outputs a single shuffled JSONL file ready for Unsloth training.


In [12]:
TARGET_BLEND = {
    "qa": 0.60,
    "continuation": 0.40,
}

data_pools = defaultdict(list)

qa_source = OUTPUT_FILE
if os.path.exists(qa_source):
    with open(qa_source) as f:
        for line in f:
            entry = json.loads(line)
            entry["data_type"] = "qa"
            data_pools["qa"].append(entry)
    print(f"  Q/A:           {len(data_pools['qa']):>6,} conversations from {qa_source}")
else:
    print(f"  ⚠ Q/A file not found: {qa_source}")

if ENABLE_CONTINUATION:
    cont_files = sorted(glob.glob(f"{CONTINUATION_DIR}/*_continuation.jsonl"))
    for cf in cont_files:
        with open(cf) as f:
            for line in f:
                entry = json.loads(line)
                entry.setdefault("data_type", "continuation")
                data_pools["continuation"].append(entry)
    print(f"  Continuation:  {len(data_pools['continuation']):>6,} entries from {len(cont_files)} files")

qa_count = len(data_pools.get("qa", []))
if qa_count == 0:
    raise RuntimeError("No Q/A data found. Run sections 5–6 first.")

active_types = {k: v for k, v in TARGET_BLEND.items() if k in data_pools and len(data_pools[k]) > 0}
total_target = int(qa_count / active_types.get("qa", 0.6))

print(f"\n  Target total:  {total_target:>6,} conversations")
print(f"  Active blend:  {active_types}")

combined = []
for dtype, fraction in active_types.items():
    pool = data_pools[dtype]
    target_count = int(total_target * fraction)
    if len(pool) >= target_count:
        sampled = random.sample(pool, target_count)
        action = "downsampled"
    else:
        repeats = target_count // len(pool)
        remainder = target_count % len(pool)
        sampled = pool * repeats + random.sample(pool, remainder)
        action = f"upsampled ({repeats}x + {remainder})"
    combined.extend(sampled)
    print(f"  {dtype:15s} {len(pool):>6,} available → {len(sampled):>6,} selected ({action})")

random.shuffle(combined)

with open(COMBINED_OUTPUT_FILE, "w") as f:
    for entry in combined:
        f.write(json.dumps(entry) + "\n")

file_size_mb = os.path.getsize(COMBINED_OUTPUT_FILE) / 1024 / 1024
print(f"\n✓ Combined training file saved:")
print(f"  {COMBINED_OUTPUT_FILE}")
print(f"  {len(combined):,} conversations ({file_size_mb:.1f} MB)")

subprocess.run(["shuf", COMBINED_OUTPUT_FILE, "-o", COMBINED_OUTPUT_FILE])
print(f"  ✓ Shuffled with shuf")

del combined, data_pools
gc.collect()


  Q/A:           15,140 conversations from /workspace/training/stoic/data/training-data/stoic_persona/stoic_personas_sharegpt.jsonl
  Continuation:   2,596 entries from 4 files

  Target total:  25,233 conversations
  Active blend:  {'qa': 0.6, 'continuation': 0.4}
  qa              15,140 available → 15,139 selected (downsampled)
  continuation     2,596 available → 10,093 selected (upsampled (3x + 2305))

✓ Combined training file saved:
  /workspace/training/stoic/data/training-data/stoic_persona/stoic_personas_combined_sharegpt.jsonl
  25,232 conversations (164.3 MB)
  ✓ Shuffled with shuf


0

## 10. Verify Combined Dataset

In [13]:
type_counts = Counter()
persona_by_type = defaultdict(lambda: defaultdict(int))
turn_dist = Counter()
format_errors = []
samples_by_type = {}
total_entries = 0

with open(COMBINED_OUTPUT_FILE) as f:
    for line_num, line in enumerate(f):
        entry = json.loads(line)
        total_entries += 1
        dtype = entry.get("data_type", "qa")
        type_counts[dtype] += 1

        convs = entry.get("conversations", [])
        turn_dist[len(convs)] += 1

        if len(convs) < 3:
            format_errors.append((line_num, f"Too few turns: {len(convs)}"))
            continue
        if convs[0]["from"] != "system":
            format_errors.append((line_num, f"First turn not system: {convs[0]['from']}"))
            continue
        for j in range(1, len(convs)):
            expected = "human" if j % 2 == 1 else "gpt"
            if convs[j]["from"] != expected:
                format_errors.append((line_num, f"Turn {j} wrong role: {convs[j]['from']} (expected {expected})"))
                break

        sys_msg = convs[0]["value"]
        for persona in PERSONA_METADATA:
            name = persona.replace("_", " ").title()
            if name in sys_msg:
                persona_by_type[dtype][persona] += 1
                break

        if dtype not in samples_by_type:
            samples_by_type[dtype] = entry

print("=" * 70)
print("COMBINED DATASET VERIFICATION")
print("=" * 70)

print(f"\nTotal entries: {total_entries:,}")
print(f"Format errors: {len(format_errors)}")
if format_errors:
    print(f"  First 5 errors:")
    for idx, msg in format_errors[:5]:
        print(f"    Line {idx}: {msg}")

print(f"\n{'─' * 50}")
print(f"DATA TYPE DISTRIBUTION")
print(f"{'─' * 50}")
print(f"{'Type':<20} {'Count':>8} {'%':>8}  Bar")
for dtype, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    pct = count / total_entries * 100
    bar = "█" * int(pct / 2)
    print(f"  {dtype:<18} {count:>7,} {pct:>7.1f}%  {bar}")
print(f"  {'TOTAL':<18} {total_entries:>7,}")

print(f"\n{'─' * 50}")
print(f"TURN COUNT DISTRIBUTION")
print(f"{'─' * 50}")
for turns, count in sorted(turn_dist.items()):
    print(f"  {turns} turns: {count:>6,}")

print(f"\n{'─' * 50}")
print(f"PERSONA DISTRIBUTION BY DATA TYPE")
print(f"{'─' * 50}")
for dtype in sorted(persona_by_type.keys()):
    persona_counts = persona_by_type[dtype]
    total_for_type = sum(persona_counts.values())
    print(f"\n  [{dtype.upper()}] ({total_for_type} total):")
    for p, c in sorted(persona_counts.items(), key=lambda x: -x[1])[:10]:
        bar = "▪" * max(1, c * 30 // max(persona_counts.values()))
        print(f"    {p:<20} {c:>5}  {bar}")

print(f"\n{'─' * 50}")
print(f"SAMPLE CONVERSATIONS (one per data type)")
print(f"{'─' * 50}")
for dtype, sample in samples_by_type.items():
    print(f"\n  ── {dtype.upper()} ──")
    for msg in sample["conversations"]:
        role = msg["from"].upper()
        text = msg["value"][:200] + ("..." if len(msg["value"]) > 200 else "")
        print(f"  [{role}] {text}\n")

print(f"\n{'=' * 70}")
if len(format_errors) == 0:
    print(f"✓ Dataset valid. Ready for training:")
    print(f"  {COMBINED_OUTPUT_FILE}")
else:
    print(f"⚠ {len(format_errors)} format errors found. Review before training.")

del type_counts, persona_by_type, turn_dist, samples_by_type
gc.collect()


COMBINED DATASET VERIFICATION

Total entries: 25,232
Format errors: 0

──────────────────────────────────────────────────
DATA TYPE DISTRIBUTION
──────────────────────────────────────────────────
Type                    Count        %  Bar
  qa                  15,139    60.0%  █████████████████████████████
  continuation        10,093    40.0%  ████████████████████
  TOTAL               25,232

──────────────────────────────────────────────────
TURN COUNT DISTRIBUTION
──────────────────────────────────────────────────
  3 turns: 10,093
  5 turns:    321
  7 turns:  3,302
  9 turns: 11,516

──────────────────────────────────────────────────
PERSONA DISTRIBUTION BY DATA TYPE
──────────────────────────────────────────────────

  [CONTINUATION] (10093 total):
    seneca                7287  ▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪▪
    epictetus             1574  ▪▪▪▪▪▪
    marcus_aurelius       1193  ▪▪▪▪
    epicurus                39  ▪

  [QA] (15139 total):
    seneca               10417  ▪▪▪▪▪

0